# Lab 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giswqs/geog-312/blob/main/book/labs/lab_03.ipynb)

This notebook contains exercises based on the lectures on [**Functions and Classes**](https://geog-312.gishub.org/book/python/06_functions_classes.html) and [**Files and Exception Handling**](https://geog-312.gishub.org/book/python/07_files.html). These exercises will help reinforce the concepts of functions, classes, file handling, and exception management in geospatial contexts.

## Exercise 1: Calculating Distances with Functions

- Define a function `calculate_distance` that takes two geographic coordinates (latitude and longitude) and returns the distance between them using the Haversine formula.
- Use this function to calculate the distance between multiple pairs of coordinates.

In [11]:
from math import radians, sqrt, sin, cos, atan2

def haversine(lat1, lon1, lat2, lon2):
  R = 6371
  dlat = radians(lat2 - lat1)
  dlon = radians(lon2 - lon1)
  a = (sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2)
  c = 2 * atan2(sqrt(a), sqrt(1 - a))
  distance = R * c
  return distance

def calculate_distance(point1, point2):
  lat1, lon1 = point1
  lat2, lon2 = point2

  d = haversine(lat1, lon1, lat2, lon2)
  return d

## Exercise 2: Batch Distance Calculation

- Create a function `batch_distance_calculation` that accepts a list of coordinate pairs and returns a list of distances between consecutive pairs.
- Test the function with a list of coordinates representing several cities.

In [12]:
def batch_distance_calculation(coordinates):
  distances_btn_points = []
  for p1, p2 in coordinates:
    dist = calculate_distance(p1, p2)
    distances_btn_points.append(dist)

  return distances_btn_points

# Test data: A list of tuples, where each tuple contains (Point A, Point B)
test_coordinates = [
    # 1. New York to Los Angeles (Long domestic)
    ((34.0522, -118.2437), (40.7128, -74.0060)),

    # 2. Paris to Tokyo (Very long intercontinental)
    ((48.8566, 2.3522), (35.6762, 139.6503)),

    # 3. Cairo to Cape Town (North-to-South Africa)
    ((30.0444, 31.2357), (-33.9249, 18.4241)),

    # 4. Tokyo to Sydney (Cross-Equator)
    ((35.6762, 139.6503), (-33.8688, 151.2093)),

    # 5. Exact same point (Zero check)
    ((10.6262, 52.65651), (10.6262, 52.65651))
]

# Run the batch code
results = batch_distance_calculation(test_coordinates)

# Print verification
for i, dist in enumerate(results, 1):
    print(f"Pair {i}: {dist:.4f} km")

Pair 1: 3935.7463 km
Pair 2: 9711.7248 km
Pair 3: 7239.2469 km
Pair 4: 7825.8186 km
Pair 5: 0.0000 km


## Exercise 3: Creating and Using a Point Class

- Define a `Point` class to represent a geographic point with attributes `latitude`, `longitude`, and `name`.
- Add a method `distance_to` that calculates the distance from one point to another.
- Instantiate several `Point` objects and calculate the distance between them.

In [13]:
class Point:
  def __init__(self, latitude, longitude, name=None):
    self.latitude = latitude
    self.longitude = longitude
    self.name = name

  def __str__(self) -> str:
     return f"{self.name or "Point"} {(self.latitude, self.longitude)}"

  def distance_to(self, point2):
    return haversine(
        self.latitude, self.longitude, point2.latitude, point2.longitude
        )


# Creating points with names
nyc = Point(40.7128, -74.0060, "New York City")
london = Point(51.5074, -0.1278, "London")

# Creating a point without a name (defaults to "Point")
mystery_spot = Point(10.6262, 52.65651)

# Testing your __str__ method by printing them
print(nyc)           # Output: New York City (40.7128, -74.006)
print(mystery_spot)  # Output: Point (10.6262, 52.65651)

print(nyc.distance_to(london))

New York City (40.7128, -74.006)
Point (10.6262, 52.65651)
5570.222179737958


## Exercise 4: Reading and Writing Files

- Write a function `read_coordinates` that reads a file containing a list of coordinates (latitude, longitude) and returns them as a list of tuples.
- Write another function `write_coordinates` that takes a list of coordinates and writes them to a new file.
- Ensure that both functions handle exceptions, such as missing files or improperly formatted data.

In [14]:
def read_coordinates(file_path):
  try:
    with open(file_path, 'r') as f:
      coords = []
      for  line in f:
        # Skip empty lines to prevent crashes
        if not line.strip():
          continue

        coord = line.strip().split(',')
        coord_tuple = (float(coord[0]), float(coord[1]))
        coords.append(coord_tuple)
    return coords

  except Exception as e:
    print(f"An error occurred while reading the file: {e}")
    return []

  finally:
        print(f"Finished processing {file_path}")

def write_coordinates(coordinates, output_file):
  try:
    with open(output_file, 'w') as outfile:
       for line in coordinates:
        lat, lon = line
        outfile.write(f"Latitude: {lat}, Longitude: {lon}\n")
    return output_file
  except Exception as e:
    print(f"An error occured while creating the file: {e}")
  finally:
    print(f"Process complete")

## Exercise 5: Processing Coordinates from a File

- Create a function that reads coordinates from a file and uses the `Point` class to create `Point` objects.
- Calculate the distance between each consecutive pair of points and write the results to a new file.
- Ensure the function handles file-related exceptions and gracefully handles improperly formatted lines.

In [16]:
def read_coords(file_path):
  coords = read_coordinates(file_path)
  point_objects = []
  for index, (lat, lon) in enumerate(coords, 1):
    # Create a unique name string using the current index
    unique_name = f'Location_{index}'

    # Instantiate your Point object
    point_obj = Point(latitude=lat, longitude=lon, name=unique_name)

    # Save it to our results list
    point_objects.append(point_obj)

  return point_objects

In [15]:
# Create a sample coordinates.txt file
sample_data = """35.6895,139.6917
34.0522,-118.2437
51.5074,-0.1278
-33.8688,151.2093
48.8566,2.3522"""

output_file = "coordinates.txt"

try:
    with open(output_file, "w") as file:
        file.write(sample_data)
    print(f"Sample file '{output_file}' has been created successfully.")
except Exception as e:
    print(f"An error occurred while creating the file: {e}")

Sample file 'coordinates.txt' has been created successfully.


## Exercise 6: Exception Handling in Data Processing

- Modify the `batch_distance_calculation` function to handle exceptions that might occur during the calculation, such as invalid coordinates.
- Ensure the function skips invalid data and continues processing the remaining data.